# Horn model evaluation plots + phase-center brute force

3つの評価フォルダ

- `Horn_fullmodel_evaluated`
- `Horn_cutmodel_evaluated`
- `Horn_lightmodel_evaluated`

から以下のデータを読み込み、比較します。

- `S11`
- `Beam`
- `Ellipticity`
- `XPD`
- `rerETheta`
- `imrETheta`

追加内容:

1. `rerETheta` / `imrETheta` を使った **phase center brute-force calculation**
2. 既存の beam width に加えて

\[
\mathrm{Ellipticity}
=
\frac{Width_{90}-Width_{0}}
     {Width_{90}+Width_{0}}
\]

を計算し、周波数に対してプロット

Plot style:

- Full: black / `Full`
- cut: red / `Quater cut`
- light: blue / `Quater Poly`

ファイルI/Oには `pathlib.Path`、Matplotlib は `fig, ax = plt.subplots(...)` を使用します。

> `ROOT` を3つの評価フォルダが置かれている親ディレクトリに変更してください。


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

C0 = 299_792_458.0

# ============================================================
# Path settings
# ============================================================
# 例:
# ROOT = Path(r"C:\Users\yourname\Desktop\horn_results")
ROOT = Path.cwd()

MODELS = {
    "full": {
        "folder": ROOT / "Horn_fullmodel_evaluated",
        "color": "black",
        "label": "Full",
    },
    "cut": {
        "folder": ROOT / "Horn_cutmodel_evaluated",
        "color": "red",
        "label": "Quater cut",
    },
    "light": {
        "folder": ROOT / "Horn_lightmodel_evaluated",
        "color": "blue",
        "label": "Quater Poly",
    },
}

TARGET_FILES = (
    "S11",
    "Beam",
    "Ellipticity",
    "XPD",
    "rerETheta",
    "imrETheta",
)

# ============================================================
# Phase-center settings
# ============================================================
THETA_MIN_DEG = -10.0
THETA_MAX_DEG = +10.0

Z_MIN_MM = -30.0
Z_MAX_MM = +30.0
DZ_MM = 0.02

# phase_z = phase0 + PHASE_SIGN * k * z * cos(theta)
# 必要に応じて -1 に変更
PHASE_SIGN = +1

# Diagnostic plot用
DIAGNOSTIC_MODEL = "full"
DIAGNOSTIC_FREQ_GHZ = 100.0


## 1. ファイル探索・読み込み

Windowsで拡張子表示がOFFの場合も考慮し、たとえば `S11` と `S11.csv` の両方に対応します。


In [ ]:
def find_data_file(folder: Path, stem: str) -> Path:
    """folder内から、ファイル名またはstemが指定名に一致するファイルを探す。"""
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    exact = folder / stem
    if exact.is_file():
        return exact

    matches = [
        p for p in folder.iterdir()
        if p.is_file() and p.stem.lower() == stem.lower()
    ]

    if not matches:
        raise FileNotFoundError(
            f"'{stem}' was not found in: {folder}"
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple files matched '{stem}' in {folder}: {matches}"
        )

    return matches[0]


def read_data_file(path: Path) -> pd.DataFrame:
    """CSV形式の評価データをDataFrameとして読み込む。"""
    return pd.read_csv(path, encoding="utf-8-sig")


def load_all_data(models: dict, target_files: tuple[str, ...]) -> dict:
    """全モデルについて対象ファイルだけを読み込む。"""
    data = {}

    for model_key, info in models.items():
        folder = info["folder"]
        data[model_key] = {}

        for stem in target_files:
            file_path = find_data_file(folder, stem)
            data[model_key][stem] = read_data_file(file_path)

            print(
                f"{model_key:>5} | {stem:<12} | "
                f"{file_path.name:<20} | shape={data[model_key][stem].shape}"
            )

    return data


data = load_all_data(MODELS, TARGET_FILES)


## 2. 読み込んだデータの確認

In [ ]:
for model_key in MODELS:
    print(f"\n===== {MODELS[model_key]['label']} =====")
    for name in TARGET_FILES:
        print(f"\n--- {name} ---")
        display(data[model_key][name].head())


## 3. S11

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_key, info in MODELS.items():
    df = data[model_key]["S11"]

    freq = df.iloc[:, 0]
    s11 = df.iloc[:, 1]

    ax.plot(
        freq,
        s11,
        color=info["color"],
        label=info["label"],
        linewidth=1.8,
    )

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("S11 [dB]")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()


## 4. Beam

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_key, info in MODELS.items():
    df = data[model_key]["Beam"]

    theta = df.iloc[:, 0]
    gain = df.iloc[:, 1]

    ax.plot(
        theta,
        gain,
        color=info["color"],
        label=info["label"],
        linewidth=1.8,
    )

ax.set_xlabel("Theta [deg]")
ax.set_ylabel("GainTotal / PeakGain [-]")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()


## 5. Beam width

元の `Ellipticity` ファイルに入っている

- `Phi = 0 deg` の 0.5-level beam width
- `Phi = 90 deg` の 0.5-level beam width

をそのまま比較します。


In [ ]:
def find_phi_width_columns(df: pd.DataFrame):
    """Ellipticityファイルから Phi=0deg / Phi=90deg の列を取得する。"""
    phi0_cols = [
        col for col in df.columns
        if "Phi='0deg'" in col
    ]
    phi90_cols = [
        col for col in df.columns
        if "Phi='90deg'" in col
    ]

    if len(phi0_cols) != 1 or len(phi90_cols) != 1:
        raise ValueError(
            "Could not uniquely identify Phi='0deg' and Phi='90deg' "
            f"columns. Columns: {list(df.columns)}"
        )

    return phi0_cols[0], phi90_cols[0]


fig, ax = plt.subplots(figsize=(7, 5))

for model_key, info in MODELS.items():
    df = data[model_key]["Ellipticity"]

    freq = df.iloc[:, 0]
    phi0_col, phi90_col = find_phi_width_columns(df)

    width_0 = df[phi0_col]
    width_90 = df[phi90_col]

    ax.plot(
        freq,
        width_0,
        color=info["color"],
        linestyle="-",
        linewidth=1.8,
    )
    ax.plot(
        freq,
        width_90,
        color=info["color"],
        linestyle="--",
        linewidth=1.8,
    )

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("Beam width at 0.5 [deg]")
ax.grid(True, alpha=0.3)

model_handles = [
    Line2D(
        [0], [0],
        color=info["color"],
        linewidth=2,
        label=info["label"],
    )
    for info in MODELS.values()
]
legend_models = ax.legend(
    handles=model_handles,
    title="Model",
    loc="best",
)
ax.add_artist(legend_models)

phi_handles = [
    Line2D(
        [0], [0],
        color="gray",
        linestyle="-",
        linewidth=2,
        label="Phi = 0 deg",
    ),
    Line2D(
        [0], [0],
        color="gray",
        linestyle="--",
        linewidth=2,
        label="Phi = 90 deg",
    ),
]
ax.legend(
    handles=phi_handles,
    title="Cut plane",
    loc="upper right",
)

fig.tight_layout()
plt.show()


## 6. Calculated Ellipticity

各周波数について

\[
\mathrm{Ellipticity}
=
\frac{Width_{90}-Width_{0}}
     {Width_{90}+Width_{0}}
\]

を計算します。


In [ ]:
ellipticity_results = {}

fig, ax = plt.subplots(figsize=(7, 5))

for model_key, info in MODELS.items():
    df = data[model_key]["Ellipticity"].copy()

    freq_col = df.columns[0]
    phi0_col, phi90_col = find_phi_width_columns(df)

    width_0 = df[phi0_col].to_numpy(dtype=float)
    width_90 = df[phi90_col].to_numpy(dtype=float)

    denominator = width_90 + width_0

    if np.any(np.isclose(denominator, 0.0)):
        raise ZeroDivisionError(
            f"Width_90 + Width_0 contains zero for model '{model_key}'."
        )

    ellipticity = (width_90 - width_0) / denominator

    result_df = pd.DataFrame(
        {
            "Frequency [GHz]": df[freq_col].to_numpy(dtype=float),
            "Width_0 [deg]": width_0,
            "Width_90 [deg]": width_90,
            "Ellipticity [-]": ellipticity,
        }
    )
    ellipticity_results[model_key] = result_df

    ax.plot(
        result_df["Frequency [GHz]"],
        result_df["Ellipticity [-]"],
        color=info["color"],
        label=info["label"],
        linewidth=1.8,
    )

ax.axhline(0.0, color="gray", linewidth=1.0, linestyle=":")
ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("Ellipticity [-]")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()


必要なら計算値を確認します。


In [ ]:
for model_key, result_df in ellipticity_results.items():
    print(f"\n===== {MODELS[model_key]['label']} =====")
    display(result_df)


## 7. XPD

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_key, info in MODELS.items():
    df = data[model_key]["XPD"]

    freq = df.iloc[:, 0]
    xpd = df.iloc[:, 1]

    ax.plot(
        freq,
        xpd,
        color=info["color"],
        label=info["label"],
        linewidth=1.8,
    )

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("XPD [dB]")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()


# 8. Phase-center brute force

各フォルダの `rerETheta` と `imrETheta` を使います。

HFSS export は以下の wide-table 形式を想定しています。

- 1列目: `Theta [deg]`
- 2列目以降: 各周波数の `re(rETheta)` または `im(rETheta)`
- E-plane として `Phi='0deg'` を使用

各周波数について候補位置 \(z\) を sweep し、

\[
\psi_z(\theta,f)
=
\psi_0(\theta,f)
+
s k z \cos\theta
\]

の theta 方向 phase standard deviation が最小になる位置を phase center とします。


In [ ]:
FREQ_RE = re.compile(r"Freq='([0-9.]+)GHz'")


def extract_retheta_wide(
    df: pd.DataFrame,
    component: str,
    phi_deg: float = 0.0,
):
    """
    HFSS wide tableから theta と周波数ごとの rETheta データを取得する。

    component:
        "re" or "im"
    """
    theta_col = df.columns[0]
    theta_deg = df[theta_col].to_numpy(dtype=float)

    component = component.lower()
    target_component = (
        "re(rETheta)" if component == "re"
        else "im(rETheta)"
    )

    phi_tag = f"Phi='{phi_deg:g}deg'"

    freq_data = {}

    for col in df.columns[1:]:
        if target_component not in col:
            continue

        # Phi情報がheaderにある場合は Phi=0deg のみ使用
        if "Phi=" in col and phi_tag not in col:
            continue

        match = FREQ_RE.search(col)
        if match is None:
            continue

        freq_ghz = float(match.group(1))
        freq_data[freq_ghz] = df[col].to_numpy(dtype=float)

    if not freq_data:
        raise ValueError(
            f"No {target_component} frequency columns were found "
            f"for {phi_tag}."
        )

    return theta_deg, freq_data


def prepare_phase_center_model(
    re_df: pd.DataFrame,
    im_df: pd.DataFrame,
):
    """1モデル分の rETheta Re/Im を phase-center calculation 用に整形する。"""
    theta_re_deg, re_data = extract_retheta_wide(
        re_df,
        component="re",
        phi_deg=0.0,
    )
    theta_im_deg, im_data = extract_retheta_wide(
        im_df,
        component="im",
        phi_deg=0.0,
    )

    if (
        len(theta_re_deg) != len(theta_im_deg)
        or not np.allclose(theta_re_deg, theta_im_deg)
    ):
        raise ValueError(
            "Theta samples in rerETheta and imrETheta do not match."
        )

    frequencies_ghz = np.array(
        sorted(set(re_data) & set(im_data)),
        dtype=float,
    )

    if len(frequencies_ghz) == 0:
        raise ValueError(
            "No common frequencies were found between "
            "rerETheta and imrETheta."
        )

    theta_mask = (
        (theta_re_deg >= THETA_MIN_DEG)
        & (theta_re_deg <= THETA_MAX_DEG)
    )

    theta_deg = theta_re_deg[theta_mask]

    if len(theta_deg) < 3:
        raise ValueError(
            "Too few theta samples in selected phase-center range."
        )

    order = np.argsort(theta_deg)
    theta_deg = theta_deg[order]
    theta_rad = np.deg2rad(theta_deg)

    return {
        "theta_all_deg": theta_re_deg,
        "theta_mask": theta_mask,
        "order": order,
        "theta_deg": theta_deg,
        "theta_rad": theta_rad,
        "re_data": re_data,
        "im_data": im_data,
        "frequencies_ghz": frequencies_ghz,
    }


phase_center_inputs = {}

for model_key in MODELS:
    phase_center_inputs[model_key] = prepare_phase_center_model(
        data[model_key]["rerETheta"],
        data[model_key]["imrETheta"],
    )

    pc_input = phase_center_inputs[model_key]

    print(
        f"{MODELS[model_key]['label']}: "
        f"{pc_input['frequencies_ghz'][0]:g} to "
        f"{pc_input['frequencies_ghz'][-1]:g} GHz, "
        f"{len(pc_input['frequencies_ghz'])} frequencies, "
        f"theta {pc_input['theta_deg'][0]:g} to "
        f"{pc_input['theta_deg'][-1]:g} deg"
    )


## 9. Phase-center solver

In [ ]:
z_candidates_mm = np.arange(
    Z_MIN_MM,
    Z_MAX_MM + 0.5 * DZ_MM,
    DZ_MM,
)
z_candidates_m = z_candidates_mm * 1e-3


def phase_center_at_frequency(pc_input: dict, freq_ghz: float):
    """1モデル・1周波数について brute force で phase center を求める。"""
    theta_mask = pc_input["theta_mask"]
    order = pc_input["order"]
    theta_rad = pc_input["theta_rad"]
    re_data = pc_input["re_data"]
    im_data = pc_input["im_data"]

    re_e = re_data[freq_ghz][theta_mask][order]
    im_e = im_data[freq_ghz][theta_mask][order]

    e_theta = re_e + 1j * im_e
    phase0 = np.unwrap(np.angle(e_theta))

    freq_hz = freq_ghz * 1e9
    k = 2.0 * np.pi * freq_hz / C0

    phase_z = (
        phase0[None, :]
        + PHASE_SIGN
        * k
        * z_candidates_m[:, None]
        * np.cos(theta_rad)[None, :]
    )

    phase_std = np.std(phase_z, axis=1)

    best_index = int(np.argmin(phase_std))
    z_pc_mm = z_candidates_mm[best_index]

    return {
        "freq_ghz": freq_ghz,
        "z_pc_mm": z_pc_mm,
        "min_phase_std_rad": phase_std[best_index],
        "min_phase_std_deg": np.rad2deg(phase_std[best_index]),
        "phase0": phase0,
        "phase_best": phase_z[best_index],
        "phase_std_vs_z": phase_std,
        "best_index": best_index,
    }


## 10. 全モデル・全周波数の phase center を計算

In [ ]:
phase_center_results = {}
phase_center_summary = []

for model_key, info in MODELS.items():
    pc_input = phase_center_inputs[model_key]
    rows = []

    for freq_ghz in pc_input["frequencies_ghz"]:
        result = phase_center_at_frequency(
            pc_input,
            float(freq_ghz),
        )

        rows.append(
            {
                "Frequency [GHz]": result["freq_ghz"],
                "Phase center z [mm]": result["z_pc_mm"],
                "Minimum phase STD [rad]": result["min_phase_std_rad"],
                "Minimum phase STD [deg]": result["min_phase_std_deg"],
            }
        )

    result_df = pd.DataFrame(rows)
    phase_center_results[model_key] = result_df

    mean_z_pc_mm = result_df["Phase center z [mm]"].mean()
    std_z_pc_mm = result_df["Phase center z [mm]"].std(ddof=0)

    phase_center_summary.append(
        {
            "Model": info["label"],
            "Mean phase center [mm]": mean_z_pc_mm,
            "STD over frequency [mm]": std_z_pc_mm,
        }
    )

phase_center_summary_df = pd.DataFrame(phase_center_summary)

display(phase_center_summary_df)

for model_key, result_df in phase_center_results.items():
    print(f"\n===== {MODELS[model_key]['label']} =====")
    display(result_df)


## 11. Phase center vs frequency

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_key, info in MODELS.items():
    result_df = phase_center_results[model_key]

    ax.plot(
        result_df["Frequency [GHz]"],
        result_df["Phase center z [mm]"],
        color=info["color"],
        label=info["label"],
        linewidth=1.8,
    )

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("Phase center z [mm]")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()


## 12. Phase-center diagnostic plot

`DIAGNOSTIC_MODEL` と `DIAGNOSTIC_FREQ_GHZ` で指定した1ケースについて、

1. phase STD vs candidate z
2. 補正前後の phase vs theta

を表示します。


In [ ]:
if DIAGNOSTIC_MODEL not in MODELS:
    raise KeyError(
        f"DIAGNOSTIC_MODEL must be one of {tuple(MODELS.keys())}"
    )

pc_input = phase_center_inputs[DIAGNOSTIC_MODEL]
frequencies_ghz = pc_input["frequencies_ghz"]

if DIAGNOSTIC_FREQ_GHZ is None:
    diagnostic_freq_ghz = frequencies_ghz[
        len(frequencies_ghz) // 2
    ]
else:
    diagnostic_freq_ghz = frequencies_ghz[
        np.argmin(
            np.abs(
                frequencies_ghz
                - DIAGNOSTIC_FREQ_GHZ
            )
        )
    ]

diag = phase_center_at_frequency(
    pc_input,
    float(diagnostic_freq_ghz),
)

print(
    f"Model: {MODELS[DIAGNOSTIC_MODEL]['label']}\n"
    f"Diagnostic frequency: {diag['freq_ghz']:g} GHz\n"
    f"z_pc = {diag['z_pc_mm']:.4f} mm\n"
    f"minimum phase STD = "
    f"{diag['min_phase_std_deg']:.4f} deg"
)

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(
    z_candidates_mm,
    np.rad2deg(diag["phase_std_vs_z"]),
    color=MODELS[DIAGNOSTIC_MODEL]["color"],
)

ax.axvline(
    diag["z_pc_mm"],
    color="gray",
    linestyle="--",
    label=f"z_pc = {diag['z_pc_mm']:.3f} mm",
)

ax.set_xlabel("Candidate z [mm]")
ax.set_ylabel("Phase STD over theta [deg]")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(
    pc_input["theta_deg"],
    np.rad2deg(diag["phase0"]),
    color="black",
    linestyle="--",
    label="Original HFSS origin",
)

ax.plot(
    pc_input["theta_deg"],
    np.rad2deg(diag["phase_best"]),
    color=MODELS[DIAGNOSTIC_MODEL]["color"],
    label=f"Shifted to z = {diag['z_pc_mm']:.3f} mm",
)

ax.set_xlabel("Theta [deg]")
ax.set_ylabel("Unwrapped phase [deg]")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()


## 13. Optional: 結果保存

必要なら以下のように計算結果をCSV保存できます。


In [ ]:
# OUTPUT_DIR = ROOT / "processed_results"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#
# for model_key in MODELS:
#     ellipticity_results[model_key].to_csv(
#         OUTPUT_DIR / f"{model_key}_ellipticity_calculated.csv",
#         index=False,
#     )
#
#     phase_center_results[model_key].to_csv(
#         OUTPUT_DIR / f"{model_key}_phase_center.csv",
#         index=False,
#     )
#
# phase_center_summary_df.to_csv(
#     OUTPUT_DIR / "phase_center_summary.csv",
#     index=False,
# )


## Notes

- Phase-center calculation は E-plane (`Phi = 0 deg`) の `rETheta` を使用します。
- optimum が `Z_MIN_MM` または `Z_MAX_MM` に張り付く場合は search range を広げてください。
- HFSS の位相規約と z 軸定義によって phase-center の符号が逆になる場合は `PHASE_SIGN = -1` に変更してください。
- Broadband phase-center stability は `STD over frequency [mm]` で確認できます。
- Ellipticity はここでは指定式 `(Width_90 - Width_0) / (Width_90 + Width_0)` の無次元量として扱っています。
